# 04 - Guidance training (P7.4)

3 platforms x 2 arms x 3 seeds = 18 runs.

* A3: APN + residual RL, near-zero action initialisation, asymmetric critic.
* A4: pure RL with the same environment-step budget as A3.

Both arms use domain randomisation and the evasive target set.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
print('repo root:', ROOT)


In [ ]:
from training.algos import load_training_config
from training.runner import train
cfg = load_training_config()
BUDGET = int(cfg['budgets']['guidance_train_steps'])
BUDGET

In [ ]:
PLATFORMS = ('quadcopter', 'fixed_wing', 'missile')
ARMS = {'A3': 'apn_rl', 'A4': 'pure_rl'}
for platform in PLATFORMS:
    for arm, mode in ARMS.items():
        for seed in (0, 1, 2):
            run_id = f'p7_{platform}_{arm.lower()}_s{seed}'
            out = train(run_id, {
                'task': 'guidance', 'algo': 'ppo', 'seed': seed,
                'total_steps': BUDGET, 'checkpoint_every': 25000,
                'asymmetric': True, 'actor_obs_dim': 85,
                'guidance': {'platform': platform, 'mode': mode,
                             'target_set': 'evasive',
                             'noise_scale': 1.0}})
            print(run_id, out['num_timesteps'])

The near-zero residual initialisation is the last layer zero-initialisation of the actor in SB3's default orthogonal initialisation; the APN command is the base action and the policy learns a bounded residual.